In [1]:
import pandas as pd
test_data = pd.read_parquet("/home/omer_ahmed/Experiments/Technique_classification/processed_span_data/test.parquet")
train_data = pd.read_parquet("/home/omer_ahmed/Experiments/Technique_classification/processed_span_data/train.parquet")
val_data = pd.read_parquet("/home/omer_ahmed/Experiments/Technique_classification/processed_span_data/val.parquet")


In [4]:
val_data

,article_id,span_start,span_end,article_text,span_text,techniques
0,111111111,149,157,Next plague outbreak in Madagascar could be 's...,appeared,[Doubt]
1,111111111,265,323,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,[Appeal_to_Authority]
2,111111111,1069,1091,Next plague outbreak in Madagascar could be 's...,"a very, very different",[Repetition]
3,111111111,1334,1462,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,[Appeal_to_fear-prejudice]
4,111111111,1577,1616,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,[Appeal_to_fear-prejudice]
...,...,...,...,...,...,...
1509,11448,208,236,One of the images frequently posted for these ...,tells her they’re targeting,"[Bandwagon,Reductio_ad_hitlerum]"
1510,11489,142,197,Prisoners either put the clothes in water to e...,"or smoke or vape the ripped-up, drug-drenched ...",[Loaded_Language]
1511,11503,100,106,The government's own Prisons Inspectorate has ...,safety,[Appeal_to_fear-prejudice]
1512,11511,0,76,"But soon, Coleman would later tell the FBI, th...","But soon, Coleman would later tell the FBI, th...","[Whataboutism,Straw_Men,Red_Herring]"


In [1]:
from model import *
from evaluation_helper import evaluate_full_pipeline
import pandas as pd
test_data = pd.read_parquet("/home/omer_ahmed/Experiments/Technique_classification/processed_span_data/test.parquet")

# IMPORTANT (same fix as training)
test_data["techniques"] = test_data["techniques"].apply(parse_labels)
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

# rebuild label map (same as training)
label2id, id2label = build_label_map(test_data)

model = TechniqueClassifier(CFG.model_name, len(label2id))
checkpoint = torch.load("best_technique_model.pt")

model.load_state_dict(checkpoint["model_state_dict"])

label2id = checkpoint["label2id"]
id2label = checkpoint["id2label"]
model.to(CFG.device)
evaluate_full_pipeline(
    model=model,
    test_df=test_data,
    dataset_class=TechniqueDataset,
    tokenizer=tokenizer,
    label2id=label2id,
    id2label=id2label,
    scorer_script_path="task-TC_scorer.py",
    techniques_list_path="propaganda-techniques-names-semeval2020task11.txt",
    output_dir="eval_results",
    device=CFG.device
)

/home/omer_ahmed/Experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total labels: 14


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4116.16it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Gold file saved to: eval_results/gold.tsv
Dataset built: 1661/1661 positive samples


Predicting: 100%|██████████| 208/208 [00:35<00:00,  5.91it/s]


✅ Submission file saved to: eval_results/submission.tsv

🚀 Running SemEval scorer...

2026-04-22 22:28:15,997 - INFO - Checking format: User Predictions -- Gold Annotations
2026-04-22 22:28:15,998 - INFO - OK: submission file format appears to be correct
2026-04-22 22:28:16,030 - INFO - Scoring submission
F1=0.883413
Precision=0.883413
Recall=0.883413
F1_Appeal_to_Authority=0.6792452830188679
F1_Appeal_to_fear-prejudice=0.8467153284671532
F1_Bandwagon,Reductio_ad_hitlerum=0.71875
F1_Black-and-White_Fallacy=0.8372093023255814
F1_Causal_Oversimplification=0.8842105263157894
F1_Doubt=0.9027777777777778
F1_Exaggeration,Minimisation=0.8537549407114624
F1_Flag-Waving=0.9218106995884774
F1_Loaded_Language=0.9362445414847161
F1_Name_Calling,Labeling=0.9114470842332614
F1_Repetition=0.8284023668639053
F1_Slogans=0.7884615384615384
F1_Thought-terminating_Cliches=0.75
F1_Whataboutism,Straw_Men,Red_Herring=0.6




In [ ]:
import torch
ckpt = torch.load("best_span_roberta_pos_ner_discourse_updated.pt")

print(ckpt["cfg"])

{'model_name': 'roberta-large', 'max_length': 512, 'stride': 384, 'batch_size': 2, 'lr': 2e-05, 'weight_decay': 0.01, 'epochs': 5, 'warmup_ratio': 0.1, 'num_workers': 2, 'seed': 42, 'lstm_hidden': 512, 'lstm_layers': 1, 'pos_dim': 32, 'ner_dim': 32, 'discourse_dim': 16, 'discourse_input_dim': 11, 'dropout': 0.2, 'save_path': 'best_span_roberta_pos_ner_discourse.pt', 'class_weights': (1.0, 1.0, 1.0, 1.0, 1.0), 'ce_loss_weight': 0.0, 'crf_loss_weight': 1.0, 'min_token_overlap_ratio': 0.0, 'pos_scale': 0.3, 'ner_scale': 0.3, 'discourse_scale': 0.3}
